# PDf Load ->Chunking (hybrid) -> Convert Embeddings->Store VDB

In [17]:
#PDF Loader
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("attention-is-all-you-need-Paper.pdf")
documents = loader.load()

#print(documents[0].page_content)


In [18]:
#Chunking using RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, 
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""])
chunks  = text_splitter.split_documents(documents)
print(f"Number of chunks: {len(chunks)}")
#print(f"First chunk: {chunks[0].page_content}")

Number of chunks: 55


In [19]:
#convert chunks to embeddings 
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [chunk.page_content for chunk in chunks]
embeddings=model.encode(texts)
print(f"Embedding of first chunk: {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5148.04it/s]


Embedding of first chunk: (55, 384)


#Vector DB uses ANN indexes.-:Approximate Nearest Neighbor
Not:100% exact
But:99.9% accurate
100x faster

Used by:
FAISS
Quadrent

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


# Create (or open) the Chroma collection and persist it
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="attention_paper",
    persist_directory="./vector_db"
)

# Example query
query = "What is self-attention?"
results = vectorstore.similarity_search(query, k=5)
for i, doc in enumerate(results):
    print(f"Result {i+1}:", doc.page_content[:300].replace('\n', ' '))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2568.04it/s]


Result 1: it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as descri
Result 2: considerably, toO(k·n·d +n·d2). Even with k = n, however, the complexity of a separable convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer, the approach we take in our model. As side beneﬁt, self-attention could yield more interpretable models. We i
Result 3: and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [31, 2, 8]. • The encoder contains sel
Result 4: PE pos. We also experimented with using learned positiona

ModuleNotFoundError: No module named 'langchain.chains'